# Station Nord 2024

Clear-sky solar and thermal irradiance at Station Nord (Villum Research Station, Greenland, 81.6°N) from the radiosondes launched during the 2024 campaign. Each sounding is written as a libRadtran radiosonde file (pressure, temperature, relative humidity) and simulated at its launch time ({doc}`/guide/atmosphere`); {doc}`nya_rad` does the same with a public IGRA sounding.

```{note}
The Station Nord radiosonde files are campaign data on `/projekt_agmwend` and are not public. This notebook is not executed for the book and has no stored outputs; run it with the campaign data or point `RS_DIR` to your own soundings. The variable names (`air_pressure`, `air_temperature`, `relative_humidity`) are those of the campaign files; adapt them for other data.
```

In [ ]:
import glob
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import pyradtran
from pyradtran import Var, load_config

logging.getLogger('pyradtran').setLevel(logging.CRITICAL)
Path("work").mkdir(exist_ok=True)

lat, lon = 81.60, -16.67    # Station Nord
RS_DIR = "/projekt_agmwend/home_rad/Joshua/STN2024/RS_STN/radiosonde_netcdf"

configs = {}
for source, wl, mol in [("solar", [290, 4000], "reptran coarse"), ("thermal", [2500, 100000], "reptran coarse")]:
    cfg = load_config()
    cfg.simulation_defaults.rte_solver = "disort"
    cfg.simulation_defaults.mol_abs_param = mol
    cfg.simulation_defaults.source = source
    cfg.simulation_defaults.wavelength_nm = wl
    cfg.simulation_defaults.integrate_wavelength = True
    cfg.simulation_defaults.albedo_value = 0.8 if source == "solar" else 0.01   # snow; emissivity 0.99
    cfg.simulation_defaults.output_columns = ["zout", "sza", "edir", "edn", "eup"]
    cfg.execution.max_workers = 8
    configs[source] = Path(f"work/stn2024_{source}.yaml")
    cfg.to_yaml(configs[source])

## Radiosonde files

One file per sounding, top of the profile first, as the uvspec `radiosonde ... H2O RH` option expects. Pressure is converted from Pa and temperature from °C when needed. Above the top of the sounding uvspec uses the background atmosphere.

In [ ]:
def write_sonde(nc_path, out_path):
    """Write one sounding as a libRadtran radiosonde file; return its launch time."""
    s = xr.open_dataset(nc_path)
    df = s[['air_pressure', 'air_temperature', 'relative_humidity']].to_dataframe().dropna()
    if df.air_pressure.max() > 2000:        # Pa -> hPa
        df['air_pressure'] /= 100
    if df.air_temperature.max() < 150:      # °C -> K
        df['air_temperature'] += 273.15
    df = df.drop_duplicates('air_pressure').sort_values('air_pressure')   # top first
    np.savetxt(out_path, df[['air_pressure', 'air_temperature', 'relative_humidity']].values,
               fmt='%.2f', header='p(hPa)  T(K)  RH(%)')
    return pd.Timestamp(s['time'].values.min())


launches, files = [], []
for i, f in enumerate(sorted(glob.glob(f"{RS_DIR}/*.nc"))):
    out = f"work/stn2024_sonde_{i:03d}.dat"
    launches.append(write_sonde(f, out))
    files.append(f"{out} H2O RH")
print(f"{len(files)} soundings, {launches[0]} to {launches[-1]}")

## Simulations

One point per sounding at its launch time. `Var('sonde')` hands every point its own radiosonde file ({doc}`/guide/parameters`).

In [ ]:
ds = xr.Dataset(
    coords={'time': pd.DatetimeIndex(launches)},
    data_vars={
        'latitude': ('time', np.full(len(launches), lat)),
        'longitude': ('time', np.full(len(launches), lon)),
        'sonde': ('time', files),
    },
).sortby('time')

params = {'radiosonde': Var('sonde')}
ds_solar = ds.pyradtran.run(config_path=configs['solar'], params=params, save_to_file=False, show_progress=False)
ds_thermal = ds.pyradtran.run(
    config_path=configs['thermal'],
    params={**params, 'sur_temperature': 253.15},    # assumption; use the measured skin temperature if available
    save_to_file=False, show_progress=False,
)

## Results

Downwelling irradiance at the surface. Solar output is in mW m⁻² and divided by 1000 ({doc}`/guide/outputs`); points with the Sun below the horizon have no solar value.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
((ds_solar.edir + ds_solar.edn).isel(altitude=0) / 1000).plot(ax=ax[0], marker='o')
(ds_thermal.edn.isel(altitude=0)).plot(ax=ax[1], marker='o')
for a, t in zip(ax, ['Solar', 'Thermal']):
    a.text(0.02, 0.85, t, transform=a.transAxes, fontsize=12, fontweight='bold')
    a.set_ylabel('Downwelling irradiance (W/m²)')
    a.set_title('')
    a.grid()
    a.spines[['top', 'right']].set_visible(False)